<a href="https://colab.research.google.com/github/LucasLisboaDev/TalentMatch-AI/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Exploration & Preprocessing

**Project:** Resume-JD Scorer — Fine-tuned DistilBERT  
**Goal of this notebook:** Load the `netsol/resume-score-details` dataset, understand the schema, clean invalid samples, normalize scores to 0-100, and save a clean DataFrame ready for training.

---

## Why this dataset?
- 1,031 resume-JD pairs labeled by GPT-4o
- Scores come from a structured rubric (macro + micro criteria)
- Gives us richer signal than simple binary match/no-match labels

## What we'll do
1. Install dependencies
2. Load the raw dataset
3. Inspect the schema
4. Filter invalid samples
5. Normalize scores to 0-100
6. Analyze score distribution
7. Save clean CSV for training


In [1]:
# Step 1 — Install dependencies
# Run this cell first every time you open this notebook in Colab
!pip install -q datasets transformers pandas numpy matplotlib seaborn

In [ ]:
# Step 2 — Load the raw dataset from HuggingFace Hub
from datasets import load_dataset

dataset = load_dataset("netsol/resume-score-details")
print(dataset)
print(f"\nTotal samples: {len(dataset['train'])}")

In [ ]:
# Step 3 — Inspect the schema
# Look at the first sample to understand what fields are available
import json

sample = dataset['train'][0]
print("Keys in a sample:")
print(list(sample.keys()))
print("\n--- Full sample (pretty printed) ---")
print(json.dumps(sample, indent=2, default=str)[:3000])  # truncate for readability

In [ ]:
# Step 4 — Extract and clean
# We pull: resume_text, jd_text, aggregated macro+micro scores, valid flag
# Samples flagged invalid or missing scores are dropped

import pandas as pd
import numpy as np

def extract_fields(sample):
    try:
        output = sample.get("output", {})
        if isinstance(output, str):
            output = json.loads(output)

        scores = output.get("scores", {})
        aggregated = scores.get("aggregated_scores", {})

        resume_text = sample.get("resume_text", "") or ""
        jd_text = sample.get("jd_text", "") or ""
        macro_score = aggregated.get("macro_scores", None)
        micro_score = aggregated.get("micro_scores", None)
        valid = output.get("valid_resume_and_jd", False)

        if not valid or not resume_text.strip() or not jd_text.strip():
            return None
        if macro_score is None or micro_score is None:
            return None

        return {
            "resume": resume_text.strip(),
            "jd": jd_text.strip(),
            "macro_score": float(macro_score),
            "micro_score": float(micro_score),
        }
    except Exception as e:
        return None

records = []
skipped = 0
for sample in dataset['train']:
    extracted = extract_fields(sample)
    if extracted is None:
        skipped += 1
        continue
    records.append(extracted)

print(f"Valid samples: {len(records)}")
print(f"Skipped (invalid): {skipped}")

In [ ]:
# Step 5 — Normalize scores to 0-100
# Both macro and micro scores are on a 0-5 scale from the GPT-4o rubric.
# We combine them (60% macro, 40% micro) and rescale to 0-100.
# The 60/40 weighting reflects that macro criteria (experience, strategic fit)
# matter more than micro criteria (specific tool matches) in real hiring decisions.

def normalize_score(macro, micro, scale=5.0):
    combined = (0.6 * macro) + (0.4 * micro)
    normalized = (combined / scale) * 100
    return round(float(np.clip(normalized, 0, 100)), 2)

for r in records:
    r["score"] = normalize_score(r["macro_score"], r["micro_score"])

df = pd.DataFrame(records)
print(df[["macro_score", "micro_score", "score"]].head(10))

In [ ]:
# Step 6 — Analyze score distribution
# This tells us if our dataset is balanced or skewed
# (important for training — a skewed dataset biases the model)

import matplotlib.pyplot as plt
import seaborn as sns

print("Score distribution stats:")
print(df["score"].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df["score"], bins=20, kde=True, color="steelblue")
plt.title("Resume-JD Match Score Distribution (0-100)")
plt.xlabel("Score")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Step 7 — Train / validation / test split and save
# 80% train, 10% validation, 10% test
# We shuffle first to avoid ordering bias

from sklearn.model_selection import train_test_split

df_clean = df[["resume", "jd", "score"]].copy()
df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Save to Google Drive so notebooks 02 and 03 can load them
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/resume-jd-scorer/"
import os
os.makedirs(save_path, exist_ok=True)

train_df.to_csv(f"{save_path}train.csv", index=False)
val_df.to_csv(f"{save_path}val.csv", index=False)
test_df.to_csv(f"{save_path}test.csv", index=False)

print(f"Saved to: {save_path}")

## Summary

After this notebook you should have:
- A clear understanding of the dataset schema
- Clean train/val/test CSVs saved to Google Drive
- A score distribution plot showing how balanced the dataset is

**Next step:** Open `02_full_finetune.ipynb` to train DistilBERT using the Trainer API.
